In [ ]:
# ==================================
# SALES FORECASTING & TREND ANALYSIS
# ==================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.arima.model import ARIMA

# -------------------------------
# 1. Load Dataset
# -------------------------------
# Dataset must have columns: Date, Sales
df = pd.read_csv("sales_data.csv")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

print("Dataset Loaded")

# -------------------------------
# 2. Exploratory Analysis
# -------------------------------
plt.figure(figsize=(10,5))
plt.plot(df["Date"], df["Sales"])
plt.title("Historical Sales Trend")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.show()

# -------------------------------
# 3. Create Time-Based Features
# -------------------------------
df["Month"] = df["Date"].dt.month
df["Year"] = df["Date"].dt.year

# -------------------------------
# 4. Train-Test Split
# -------------------------------
train_size = int(len(df) * 0.8)
train = df.iloc[:train_size]
test = df.iloc[train_size:]

# -------------------------------
# 5. Linear Regression Model
# -------------------------------
X_train = train[["Month", "Year"]]
y_train = train["Sales"]

X_test = test[["Month", "Year"]]
y_test = test["Sales"]

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# -------------------------------
# 6. Evaluate Linear Regression
# -------------------------------
print("\nLinear Regression Performance")
print("MAE:", mean_absolute_error(y_test, lr_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, lr_pred)))

# -------------------------------
# 7. ARIMA Model (Time Series)
# -------------------------------
arima = ARIMA(train["Sales"], order=(5,1,0))
arima_model = arima.fit()

forecast = arima_model.forecast(steps=len(test))

# -------------------------------
# 8. Evaluate ARIMA
# -------------------------------
print("\nARIMA Model Performance")
print("MAE:", mean_absolute_error(y_test, forecast))
print("RMSE:", np.sqrt(mean_squared_error(y_test, forecast)))

# -------------------------------
# 9. Plot Forecast vs Actual
# -------------------------------
plt.figure(figsize=(10,5))
plt.plot(train["Date"], train["Sales"], label="Train")
plt.plot(test["Date"], test["Sales"], label="Actual")
plt.plot(test["Date"], forecast, label="Forecast")
plt.legend()
plt.title("Sales Forecast vs Actual")
plt.show()

# -------------------------------
# 10. Save Predictions
# -------------------------------
results = pd.DataFrame({
    "Date": test["Date"],
    "Actual Sales": y_test,
    "Predicted Sales": forecast
})

results.to_csv("sales_predictions.csv", index=False)

print("\nSales Forecasting Project Completed Successfully!")
